In [0]:
from datetime import datetime, timezone
import uuid

# Initialize pipeline execution tracking
pipeline_run_id = str(uuid.uuid4())
pipeline_start_time = datetime.now(timezone.utc)

print("Pipeline Run ID:", pipeline_run_id)
print("Pipeline Start Time:", pipeline_start_time)

Pipeline Run ID: a5954247-acba-4b3b-84fc-e41ee55e11ec
Pipeline Start Time: 2026-10-31 19:58:29.090486+00:00


In [0]:
# Smart Data Quality Sentinel
# Production Pipeline

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("Smart Data Quality Sentinel - Pipeline Started")

Smart Data Quality Sentinel - Pipeline Started


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.sentinel_pipeline_logs (
    run_id STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    error_message STRING
)
USING DELTA;

In [0]:
def validate_inventory(input_df):

    window_spec = Window.partitionBy("product_id")

    checked_df = input_df.withColumn(
        "product_count",
        F.count("product_id").over(window_spec)
    )

    checked_df = checked_df.withColumn(
        "quality_issues",
        F.array_compact(F.array(
            F.when(
                F.col("supplier_id").isNull(),
                F.lit("MISSING_SUPPLIER_ID")
            ),
            F.when(
                (F.col("product_id").isNotNull()) &
                (F.col("product_count") > 1),
                F.lit("DUPLICATE_PRODUCT_ID")
            ),
            F.when(
                F.col("selling_price") < F.col("cost_price"),
                F.lit("SELLING_PRICE_BELOW_COST")
            ),
            F.when(
                F.col("quantity") < 0,
                F.lit("NEGATIVE_QUANTITY")
            ),
            F.when(
                F.datediff(
                    F.current_date(), F.col("last_updated")
                ) > 45,
                F.lit("STALE_INVENTORY")
            )
        ))
    )

    checked_df = checked_df.withColumn(
        "severity",
        F.when(
            F.array_contains(
                F.col("quality_issues"), "NEGATIVE_QUANTITY"
            ) |
            F.array_contains(
                F.col("quality_issues"), "DUPLICATE_PRODUCT_ID"
            ),
            "CRITICAL"
        ).when(
            F.size("quality_issues") > 0,
            "WARNING"
        ).otherwise("NONE")
    )

    checked_df = checked_df.withColumn(
        "quality_status",
        F.when(F.size("quality_issues") > 0, "INVALID")
         .otherwise("VALID")
    )

    return checked_df.drop("product_count")

In [0]:
def log_pipeline_execution(status, error_message=None):
    pipeline_end_time = datetime.now(timezone.utc)

    log_df = spark.createDataFrame(
        [(
            pipeline_run_id,
            pipeline_start_time,
            pipeline_end_time,
            status,
            error_message
        )],
        schema="""
            run_id STRING,
            start_time TIMESTAMP,
            end_time TIMESTAMP,
            status STRING,
            error_message STRING
        """
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.default.sentinel_pipeline_logs")
    )

    print(f"Pipeline execution logged: {status}")

In [0]:
# # Detect pending files for quality processing

# INGESTION_TRACKING_TABLE = "workspace.default.sentinel_file_tracking"
# QUALITY_TRACKING_TABLE = "workspace.default.sentinel_quality_file_tracking"
# BRONZE_TABLE = "workspace.default.sentinel_bronze_inventory"

# ingested_files_df = (
#     spark.table(INGESTION_TRACKING_TABLE)
#     .filter(F.col("ingestion_status") == "SUCCESS")
#     .filter(F.col("file_id").isNotNull())
#     .select("file_id", "file_name", "records_ingested")
#     .dropDuplicates(["file_id"])
# )

# processed_files_df = (
#     spark.table(QUALITY_TRACKING_TABLE)
#     .filter(F.col("processing_status") == "SUCCESS")
#     .select("file_id")
#     .dropDuplicates(["file_id"])
# )

# pending_files_df = ingested_files_df.join(
#     processed_files_df,
#     on="file_id",
#     how="left_anti"
# )

# pending_file_count = pending_files_df.count()

# print("Pending files:", pending_file_count)

# display(pending_files_df)

In [0]:

# # DBTITLE 1,Incrementally validate and merge records into Silver

# SILVER_TABLE = "workspace.default.sentinel_silver_inventory"

# pending_file_ids_df = pending_files_df.select("file_id").distinct()

# if pending_files_df.isEmpty():

#     print("No pending files. Silver table remains unchanged.")

# else:

#     # 1. Read Bronze records belonging to pending files
#     bronze_df = spark.table(BRONZE_TABLE)

#     new_file_records_df = bronze_df.join(
#         pending_file_ids_df,
#         bronze_df["_source_file_id"] == pending_file_ids_df["file_id"],
#         "inner"
#     ).drop("file_id")

#     # 2. Identify product IDs affected by the new files
#     affected_product_ids_df = (
#         new_file_records_df
#         .select("product_id")
#         .filter(F.col("product_id").isNotNull())
#         .distinct()
#     )

#     # 3. Find ALL Bronze records for those product IDs
#     # This includes older records that may now be duplicates.
#     related_records_df = bronze_df.join(
#         affected_product_ids_df,
#         on="product_id",
#         how="inner"
#     )

#     # Include new records with null product IDs too
#     affected_records_df = (
#         related_records_df
#         .unionByName(new_file_records_df)
#         .dropDuplicates(["record_id"])
#     )

#     print("New file records:", new_file_records_df.count())
#     print("Records requiring validation:", affected_records_df.count())

#     # 4. Validate affected records
#     validated_incremental_df = validate_inventory(affected_records_df)

#     validated_incremental_df.groupBy(
#         "quality_status", "severity"
#     ).count().show()

#     # 5. Merge validated records into Silver
#     silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

#     (
#         silver_delta.alias("target")
#         .merge(
#             validated_incremental_df.alias("source"),
#             "target.record_id = source.record_id"
#         )
#         .whenMatchedUpdateAll()
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

#     print("Incremental Silver MERGE completed.")


In [0]:
# DBTITLE 1,Execute Incremental Data Quality Pipeline

try:

    # ============================================================
    # PIPELINE START
    # ============================================================
    # This is the main execution block of Smart Data Quality Sentinel.
    #
    # The pipeline performs the following operations:
    # 1. Identify files that have been ingested but not quality-processed.
    # 2. Read only the new records from Bronze.
    # 3. Identify all related records that may be affected by duplicates.
    # 4. Validate the affected records using our quality rules.
    # 5. MERGE validated records into Silver.
    # 6. Refresh Gold tables using the complete Silver dataset.
    # 7. Calculate overall quality and issue metrics.
    # 8. Record quality history.
    # 9. Update file processing status.
    # 10. Log pipeline execution status.
    #
    # The try/except block ensures that an exception during processing
    # is captured and recorded in the pipeline execution log.

    print("Starting Smart Data Quality Sentinel pipeline")
    print("Pipeline Run ID:", pipeline_run_id)


    # ============================================================
    # STEP 1: DEFINE SOURCE AND TARGET TABLES
    # ============================================================
    # We store table names in variables instead of repeating
    # fully qualified table names throughout the code.
    #
    # INGESTION_TRACKING_TABLE:
    # Stores information about files successfully ingested into Bronze.
    #
    # QUALITY_TRACKING_TABLE:
    # Stores information about files successfully processed
    # by the data quality pipeline.
    #
    # BRONZE_TABLE:
    # Contains raw inventory records received from uploaded CSV files.
    #
    # SILVER_TABLE:
    # Contains inventory records after applying data quality rules.

    INGESTION_TRACKING_TABLE = "workspace.default.sentinel_file_tracking"
    QUALITY_TRACKING_TABLE = "workspace.default.sentinel_quality_file_tracking"
    BRONZE_TABLE = "workspace.default.sentinel_bronze_inventory"
    SILVER_TABLE = "workspace.default.sentinel_silver_inventory"


    # ============================================================
    # STEP 2: IDENTIFY SUCCESSFULLY INGESTED FILES
    # ============================================================
    # Read the ingestion tracking table.
    #
    # We only select files with ingestion_status = SUCCESS because
    # failed or incomplete ingestion files must not enter the
    # quality-processing stage.
    #
    # file_id is used to uniquely identify the file.
    # file_name is retained for tracking and troubleshooting.
    # records_ingested tells us how many records came from the file.
    #
    # dropDuplicates prevents the same file from appearing multiple
    # times in our pending-file detection logic.

    ingested_files_df = (
        spark.table(INGESTION_TRACKING_TABLE)
        .filter(F.col("ingestion_status") == "SUCCESS")
        .filter(F.col("file_id").isNotNull())
        .select("file_id", "file_name", "records_ingested")
        .dropDuplicates(["file_id"])
    )


    # ============================================================
    # STEP 3: IDENTIFY FILES ALREADY PROCESSED BY QUALITY PIPELINE
    # ============================================================
    # Read the quality tracking table.
    #
    # We only consider files with processing_status = SUCCESS.
    #
    # A file marked SUCCESS has already completed quality processing.
    # Such files should not be processed again during normal execution.
    #
    # We select only file_id because that is the field required
    # to compare ingestion and quality-processing records.

    processed_files_df = (
        spark.table(QUALITY_TRACKING_TABLE)
        .filter(F.col("processing_status") == "SUCCESS")
        .select("file_id")
        .distinct()
    )


    # ============================================================
    # STEP 4: FIND PENDING FILES USING LEFT ANTI JOIN
    # ============================================================
    # Compare successfully ingested files with successfully
    # quality-processed files.
    #
    # A LEFT ANTI JOIN returns records from the left DataFrame
    # that do not have a matching record in the right DataFrame.
    #
    # In our case:
    # Left side  = Successfully ingested files.
    # Right side = Successfully quality-processed files.
    #
    # Result = Files that are ingested but not yet quality-processed.
    #
    # These are the files that require incremental processing.

    pending_files_df = ingested_files_df.join(
        processed_files_df,
        on="file_id",
        how="left_anti"
    )

    # Count pending files so the pipeline can decide whether
    # new data processing is required.

    pending_count = pending_files_df.count()

    print("Pending files:", pending_count)

    processing_key = None

    if pending_count > 0:
        processing_key = (
            pending_files_df
            .agg(
                F.sha2(
                    F.concat_ws(
                        "|",
                        F.sort_array(F.collect_set("file_id"))
                    ),
                    256
                ).alias("processing_key")
            )
            .first()["processing_key"]
        )

    print("Processing key:", processing_key)

    # ============================================================
    # STEP 5: PROCESS PENDING FILES
    # ============================================================
    # We only execute the data-processing logic when at least
    # one pending file is available.
    #
    # If pending_count = 0:
    # No new file needs quality processing.
    #
    # If pending_count > 0:
    # Continue with incremental Bronze-to-Silver processing.

    if pending_count > 0:


        # --------------------------------------------------------
        # STEP 5.1: READ BRONZE INVENTORY DATA
        # --------------------------------------------------------
        # Bronze contains the complete raw inventory dataset,
        # including records from previously ingested files.
        #
        # We read Bronze because we need:
        # 1. New records belonging to pending files.
        # 2. Existing records that share product IDs with new records.
        #
        # Reading Bronze does not mean we validate every record.
        # We use it to identify only the records affected by new files.

        bronze_df = spark.table(BRONZE_TABLE)


        # --------------------------------------------------------
        # STEP 5.2: EXTRACT FILE IDs THAT REQUIRE PROCESSING
        # --------------------------------------------------------
        # Get the unique file IDs from the pending-file DataFrame.
        #
        # These IDs identify which Bronze records belong to
        # files that have not completed quality processing.

        pending_file_ids_df = pending_files_df.select("file_id")


        # --------------------------------------------------------
        # STEP 5.3: READ ONLY THE NEW FILE RECORDS FROM BRONZE
        # --------------------------------------------------------
        # Match Bronze records against the pending file IDs.
        #
        # _source_file_id in Bronze identifies the file from which
        # each inventory record originated.
        #
        # The INNER JOIN returns only records belonging to
        # pending files.
        #
        # We remove the duplicate file_id column introduced by
        # the join because Bronze already has _source_file_id.

        new_file_records_df = bronze_df.join(
            pending_file_ids_df,
            bronze_df["_source_file_id"] == pending_file_ids_df["file_id"],
            "inner"
        ).drop("file_id")


        # --------------------------------------------------------
        # STEP 5.4: IDENTIFY PRODUCT IDs AFFECTED BY NEW FILES
        # --------------------------------------------------------
        # Our duplicate validation rule checks whether a product_id
        # appears more than once in the dataset.
        #
        # Therefore, validating only new records is insufficient.
        #
        # Example:
        # Existing Bronze record: product_id = 5001
        # New Bronze record:      product_id = 5001
        #
        # Both records must be identified as duplicates.
        #
        # We extract distinct, non-null product IDs from the new
        # records to identify all potentially affected inventory rows.

        affected_product_ids_df = (
            new_file_records_df
            .select("product_id")
            .filter(F.col("product_id").isNotNull())
            .distinct()
        )


        # --------------------------------------------------------
        # STEP 5.5: FIND ALL BRONZE RECORDS FOR AFFECTED PRODUCTS
        # --------------------------------------------------------
        # Search the complete Bronze dataset for records whose
        # product_id matches one of the affected product IDs.
        #
        # This retrieves both:
        # - Existing records that were ingested earlier.
        # - New records belonging to pending files.
        #
        # This step ensures duplicate detection works across
        # different files, not just within the latest file.

        related_records_df = bronze_df.join(
            affected_product_ids_df,
            on="product_id",
            how="inner"
        )


        # --------------------------------------------------------
        # STEP 5.6: COMBINE ALL RECORDS REQUIRING VALIDATION
        # --------------------------------------------------------
        # Combine:
        # 1. All records sharing affected product IDs.
        # 2. All new file records.
        #
        # The second DataFrame is included separately so that
        # new records with a NULL product_id are not missed.
        #
        # unionByName combines DataFrames using their column names.
        #
        # dropDuplicates(["record_id"]) prevents the same inventory
        # record from appearing twice in the combined dataset.
        #
        # record_id is expected to be unique for each inventory row.

        # affected_records_df = (
        #     related_records_df
        #     .unionByName(new_file_records_df)
        #     .dropDuplicates(["record_id"])
        # )

        record_window = (
            Window
            .partitionBy("record_id")
            .orderBy(F.col("_ingested_at").desc())
        )

        affected_records_df = (
            related_records_df
            .unionByName(new_file_records_df)
            .withColumn("_row_num", F.row_number().over(record_window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )

        print("New file records:", new_file_records_df.count())
        print("Records requiring validation:", affected_records_df.count())


        # --------------------------------------------------------
        # STEP 5.7: APPLY DATA QUALITY VALIDATION RULES
        # --------------------------------------------------------
        # Pass the affected records to our existing
        # validate_inventory() function.
        #
        # The function checks:
        # - Missing supplier IDs.
        # - Duplicate product IDs.
        # - Selling price below cost price.
        # - Negative inventory quantity.
        # - Stale inventory records.
        #
        # It also generates:
        # - quality_issues
        # - severity
        # - quality_status
        #
        # The validation function is reused instead of writing
        # the same quality rules again.

        validated_incremental_df = validate_inventory(affected_records_df)


        # Display a summary of validation results.
        # This helps us understand how many affected records
        # are VALID or INVALID and their severity levels.

        validated_incremental_df.groupBy(
            "quality_status", "severity"
        ).count().show()


        # --------------------------------------------------------
        # STEP 5.8: MERGE VALIDATED RECORDS INTO SILVER
        # --------------------------------------------------------
        # Silver is our cleaned and quality-validated inventory layer.
        #
        # DeltaTable.forName() provides access to the existing
        # Silver Delta table so we can perform a MERGE operation.
        #
        # MERGE matches records using record_id.
        #
        # WHEN MATCHED:
        # Update an existing Silver record with the latest
        # validated values.
        #
        # WHEN NOT MATCHED:
        # Insert the new validated record into Silver.
        #
        # This avoids overwriting the entire Silver table.
        # Only affected records are inserted or updated.

        silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

        (
            silver_delta.alias("target")
            .merge(
                validated_incremental_df.alias("source"),
                "target.record_id = source.record_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Incremental Silver MERGE completed.")


        # ========================================================
        # STEP 6: READ THE COMPLETE UPDATED SILVER DATASET
        # ========================================================
        # At this stage, Silver contains:
        # - Previously processed inventory records.
        # - Newly inserted inventory records.
        # - Existing records whose quality status was updated.
        #
        # We read the complete Silver table because Gold tables
        # must represent the current overall inventory state.
        #
        # We do not use only validated_incremental_df for Gold
        # calculations because that would produce incomplete
        # business metrics.

        silver_df = spark.table(SILVER_TABLE)

        print("Total Silver records:", silver_df.count())


        # ========================================================
        # STEP 7: REFRESH GOLD RESTOCK CANDIDATES
        # ========================================================
        # Gold Restock contains inventory products that need
        # replenishment.
        #
        # Business conditions:
        # 1. The record must have quality_status = VALID.
        # 2. Current quantity must be below reorder_level.
        #
        # shortage_quantity tells us how many units are required
        # to reach the configured reorder level.
        #
        # We calculate this using the complete Silver dataset.

        restock_candidates_df = (
            silver_df
            .filter(
                (F.col("quality_status") == "VALID") &
                (F.col("quantity") < F.col("reorder_level"))
            )
            .withColumn(
                "shortage_quantity",
                F.col("reorder_level") - F.col("quantity")
            )
            .select(
                "product_id",
                "product_name",
                "category",
                "supplier_id",
                "quantity",
                "reorder_level",
                "shortage_quantity"
            )
        )

        # Overwrite Gold Restock with the newly calculated result.
        #
        # This full refresh also removes products that no longer
        # satisfy the restocking conditions.

        (
            restock_candidates_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("workspace.default.sentinel_gold_restock")
        )

        print("Gold Restock table refreshed.")


        # ========================================================
        # STEP 8: REFRESH GOLD NEGATIVE MARGIN TABLE
        # ========================================================
        # Gold Negative Margin identifies products being sold
        # below their cost price.
        #
        # Business condition:
        # selling_price < cost_price
        #
        # loss_per_unit represents the loss incurred on each unit.
        #
        # We use the complete Silver dataset so that the Gold table
        # contains all currently identified negative-margin records.

        negative_margin_df = (
            silver_df
            .filter(F.col("selling_price") < F.col("cost_price"))
            .withColumn(
                "loss_per_unit",
                F.col("cost_price") - F.col("selling_price")
            )
            .select(
                "product_id",
                "product_name",
                "category",
                "cost_price",
                "selling_price",
                "loss_per_unit"
            )
        )

        # Replace the previous Gold Negative Margin result
        # with the latest complete calculation.

        (
            negative_margin_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("workspace.default.sentinel_gold_negative_margin")
        )

        print("Gold Negative Margin table refreshed.")


        # ========================================================
        # STEP 9: CALCULATE OVERALL DATA QUALITY METRICS
        # ========================================================
        # Calculate metrics across the complete Silver dataset.
        #
        # total_records:
        # Total number of inventory records.
        #
        # valid_records:
        # Number of records that passed all quality checks.
        #
        # invalid_records:
        # Number of records with one or more quality issues.
        #
        # critical_records:
        # Number of records classified as CRITICAL.
        #
        # warning_records:
        # Number of records classified as WARNING.
        #
        # quality_score:
        # Percentage of valid records out of total records.

        production_metrics_df = silver_df.agg(
            F.count("*").alias("total_records"),

            F.sum(
                F.when(F.col("quality_status") == "VALID", 1).otherwise(0)
            ).alias("valid_records"),

            F.sum(
                F.when(F.col("quality_status") == "INVALID", 1).otherwise(0)
            ).alias("invalid_records"),

            F.sum(
                F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)
            ).alias("critical_records"),

            F.sum(
                F.when(F.col("severity") == "WARNING", 1).otherwise(0)
            ).alias("warning_records")

        ).withColumn(
            "quality_score",
            F.when(
                F.col("total_records") > 0,
                F.round(
                    F.col("valid_records") /
                    F.col("total_records") * 100,
                    2
                )
            ).otherwise(0)
        )


        # --------------------------------------------------------
        # STEP 9.1: ADD METRIC PROCESSING TIMESTAMP
        # --------------------------------------------------------
        # processed_at records when these overall quality metrics
        # were calculated.
        #
        # This is useful when monitoring dashboard freshness.

        production_gold_metrics_df = production_metrics_df.withColumn(
            "processed_at",
            F.current_timestamp()
        )


        # --------------------------------------------------------
        # STEP 9.2: SAVE GOLD QUALITY METRICS
        # --------------------------------------------------------
        # Save the latest overall quality metrics into the Gold table.
        #
        # Overwrite ensures the table contains the latest snapshot
        # rather than accumulating outdated metric rows.

        (
            production_gold_metrics_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("workspace.default.sentinel_gold_quality_metrics")
        )

        print("Gold Quality Metrics refreshed.")


        # ========================================================
        # STEP 10: CALCULATE GOLD ISSUE METRICS
        # ========================================================
        # Each record can contain multiple quality issues.
        #
        # explode() converts the quality_issues array into
        # individual rows.
        #
        # Example:
        # ["MISSING_SUPPLIER_ID", "NEGATIVE_QUANTITY"]
        #
        # Becomes:
        # MISSING_SUPPLIER_ID
        # NEGATIVE_QUANTITY
        #
        # groupBy() groups identical issue types.
        # count() calculates how many times each issue occurs.
        #
        # This provides issue distribution for the dashboard.

        issue_metrics_df = (
            silver_df
            .select(F.explode("quality_issues").alias("issue_type"))
            .groupBy("issue_type")
            .agg(F.count("*").alias("issue_count"))
            .orderBy(F.desc("issue_count"))
        )


        # --------------------------------------------------------
        # STEP 10.1: SAVE GOLD ISSUE METRICS
        # --------------------------------------------------------
        # Replace the previous issue metrics with the latest
        # complete issue distribution.

        (
            issue_metrics_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("workspace.default.sentinel_gold_issue_metrics")
        )

        print("Gold Issue Metrics refreshed.")


        
        # ========================================================
        # STEP 11: RECORD HISTORICAL QUALITY METRICS
        # ========================================================
        # The Gold Quality Metrics table stores the latest snapshot.
        #
        # The Quality History table preserves historical snapshots.
        #
        # run_id identifies an individual pipeline execution.
        # processing_key identifies the logical batch of files.
        #
        # Using processing_key in a Delta MERGE prevents the same
        # logical batch from creating duplicate history records
        # when the pipeline is retried.

        production_history_df = (
            production_metrics_df
            .withColumn("run_id", F.lit(pipeline_run_id))
            .withColumn("run_timestamp", F.current_timestamp())
            .withColumn("processing_key", F.lit(processing_key))
            .select(
                "total_records",
                "valid_records",
                "invalid_records",
                "critical_records",
                "warning_records",
                "quality_score",
                "run_id",
                "run_timestamp",
                "processing_key"
            )
        )

        # Access the existing Delta history table.

        history_delta = DeltaTable.forName(
            spark,
            "workspace.default.sentinel_quality_history"
        )

        # Insert the history record only if this processing_key
        # does not already exist.
        #
        # If the same batch is retried, the existing record is
        # retained instead of inserting a duplicate.

        (
            history_delta.alias("target")
            .merge(
                production_history_df.alias("source"),
                "target.processing_key = source.processing_key"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Quality history recorded (idempotent MERGE).")



        # ========================================================
        # STEP 12: MARK FILES AS SUCCESSFULLY QUALITY-PROCESSED
        # ========================================================
        # Only after Silver, Gold and history operations complete
        # successfully do we mark pending files as SUCCESS.
        #
        # This prevents files from being marked complete before
        # the downstream processing has finished.
        #
        # The tracking table records:
        # file_id:
        # Unique identifier of the source file.
        #
        # file_name:
        # Original uploaded file name.
        #
        # processing_status:
        # SUCCESS indicates completed quality processing.
        #
        # records_processed:
        # Number of records ingested from the source file.
        #
        # processing_run_id:
        # Pipeline execution that processed the file.
        #
        # processed_at:
        # Timestamp when processing was completed.
        #
        # error_message:
        # NULL because processing completed successfully.

        quality_tracking_delta = DeltaTable.forName(
            spark,
            QUALITY_TRACKING_TABLE
        )


        # Prepare the tracking records for all pending files.

        tracking_updates_df = pending_files_df.select(
            "file_id",
            "file_name",

            F.lit("SUCCESS").alias("processing_status"),

            F.col("records_ingested").alias("records_processed"),

            F.lit(pipeline_run_id).alias("processing_run_id"),

            F.current_timestamp().alias("processed_at"),

            F.lit(None).cast("string").alias("error_message")
        )


        # MERGE prevents unnecessary duplicate tracking entries.
        #
        # WHEN MATCHED:
        # Update the existing file tracking record.
        #
        # WHEN NOT MATCHED:
        # Insert a new file tracking record.

        (
            quality_tracking_delta.alias("target")
            .merge(
                tracking_updates_df.alias("source"),
                "target.file_id = source.file_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("Quality file tracking updated successfully.")


    # ============================================================
    # STEP 13: HANDLE THE NO-PENDING-FILES SCENARIO
    # ============================================================
    # If no new files are available:
    # - Do not read and validate the complete Bronze dataset.
    # - Do not MERGE anything into Silver.
    # - Do not refresh Gold tables unnecessarily.
    # - Do not append duplicate quality history snapshots.
    #
    # The pipeline execution is still considered successful
    # because there was no new work to process.

    else:

        print("No pending files found.")
        print("Silver and Gold tables remain unchanged.")
        print("No quality history update required.")


    # ============================================================
    # STEP 14: LOG SUCCESSFUL PIPELINE EXECUTION
    # ============================================================
    # Reaching this point means all required operations completed
    # without raising an exception.
    #
    # Record SUCCESS in sentinel_pipeline_logs.

    log_pipeline_execution("SUCCESS")


# ================================================================
# PIPELINE FAILURE HANDLING
# ================================================================
# If any operation inside the try block fails:
# 1. Capture the exception message.
# 2. Attempt to record FAILED in the pipeline logs.
# 3. Re-raise the exception so Databricks marks the job as failed.
#
# Re-raising is important because otherwise the Databricks Job
# might incorrectly report a failed pipeline as successful.

except Exception as e:

    print("Pipeline execution failed:", str(e))

    try:

        log_pipeline_execution("FAILED", str(e))

    except Exception as logging_error:

        print("Failed to write pipeline failure log:", str(logging_error))

    raise


Starting Smart Data Quality Sentinel pipeline
Pipeline Run ID: a5954247-acba-4b3b-84fc-e41ee55e11ec
Pending files: 1
Processing key: c97db00d1c72775fffc70434660f78de8f1c7cf110ecb75e123d886f978b210a
New file records: 1
Records requiring validation: 1
+--------------+--------+-----+
|quality_status|severity|count|
+--------------+--------+-----+
|         VALID|    NONE|    1|
+--------------+--------+-----+

Incremental Silver MERGE completed.
Total Silver records: 50806
Gold Restock table refreshed.
Gold Negative Margin table refreshed.
Gold Quality Metrics refreshed.
Gold Issue Metrics refreshed.
Quality history recorded (idempotent MERGE).
Quality file tracking updated successfully.
Pipeline execution logged: SUCCESS


In [0]:
%sql
SELECT *
FROM workspace.default.sentinel_pipeline_logs
ORDER BY start_time DESC
LIMIT 5;

run_id,start_time,end_time,status,error_message
a5954247-acba-4b3b-84fc-e41ee55e11ec,2026-10-31T19:58:29.090Z,2026-10-31T19:59:00.610Z,SUCCESS,null
23674ca1-a69c-4998-9375-e2b6c7c1f556,2026-10-31T19:50:07.032Z,2026-10-31T19:50:10.154Z,SUCCESS,null
715bf21a-02f5-455d-8f22-c1fac9539906,2026-10-31T19:48:44.639Z,2026-10-31T19:49:10.584Z,SUCCESS,null
a5323729-34f1-4123-aa93-3bf3736583d6,2026-10-31T19:13:34.816Z,2026-10-31T19:14:09.249Z,SUCCESS,null
33d775e1-ecfb-418e-b927-dd4437285259,2026-10-31T17:54:02.879Z,2026-10-31T17:54:54.631Z,SUCCESS,null


In [0]:
%sql
SELECT 'Silver Inventory' AS table_name, COUNT(*) AS record_count
FROM workspace.default.sentinel_silver_inventory

UNION ALL

SELECT 'Gold Restock', COUNT(*)
FROM workspace.default.sentinel_gold_restock

UNION ALL

SELECT 'Gold Negative Margin', COUNT(*)
FROM workspace.default.sentinel_gold_negative_margin

UNION ALL

SELECT 'Gold Quality Metrics', COUNT(*)
FROM workspace.default.sentinel_gold_quality_metrics

UNION ALL

SELECT 'Gold Issue Metrics', COUNT(*)
FROM workspace.default.sentinel_gold_issue_metrics;

table_name,record_count
Silver Inventory,50806
Gold Restock,9525
Gold Negative Margin,470
Gold Quality Metrics,1
Gold Issue Metrics,5


In [0]:
%sql
SELECT *
FROM workspace.default.sentinel_gold_quality_metrics;
 

total_records,valid_records,invalid_records,critical_records,warning_records,quality_score,processed_at
50806,47915,2891,1465,1426,94.31,2026-10-31T19:58:51.659Z


In [0]:
%sql
SELECT *
FROM workspace.default.sentinel_gold_issue_metrics
ORDER BY issue_count DESC;

issue_type,issue_count
DUPLICATE_PRODUCT_ID,1001
STALE_INVENTORY,505
MISSING_SUPPLIER_ID,495
NEGATIVE_QUANTITY,476
SELLING_PRICE_BELOW_COST,470


# ---------------------------------------------------

In [0]:

# try:
#     print("Starting Smart Data Quality Sentinel pipeline")

#     # 1. Read Bronze data
#     BRONZE_TABLE = "workspace.default.sentinel_bronze_inventory"

#     bronze_df = spark.table(BRONZE_TABLE)

#     print("Bronze records:", bronze_df.count())

#     # 2. Validate inventory
#     validated_bronze_df = validate_inventory(bronze_df)

#     validated_bronze_df.groupBy(
#         "quality_status", "severity"
#     ).count().show()

#     # 3. Calculate production metrics
#     production_metrics_df = validated_bronze_df.agg(
#         F.count("*").alias("total_records"),
#         F.sum(
#             F.when(F.col("quality_status") == "VALID", 1).otherwise(0)
#         ).alias("valid_records"),
#         F.sum(
#             F.when(F.col("quality_status") == "INVALID", 1).otherwise(0)
#         ).alias("invalid_records"),
#         F.sum(
#             F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)
#         ).alias("critical_records"),
#         F.sum(
#             F.when(F.col("severity") == "WARNING", 1).otherwise(0)
#         ).alias("warning_records")
#     ).withColumn(
#         "quality_score",
#         F.when(
#             F.col("total_records") > 0,
#             F.round(
#                 F.col("valid_records") / F.col("total_records") * 100,
#                 2
#             )
#         ).otherwise(0)
#     )

#     # 4. Write Silver table
#     (
#         validated_bronze_df.write
#         .format("delta")
#         .mode("overwrite")
#         .option("overwriteSchema", "true")
#         .saveAsTable("workspace.default.sentinel_silver_inventory")
#     )

#     print("Silver table updated successfully")

#     # 5. Generate Gold restock candidates
#     restock_candidates_df = (
#         validated_bronze_df
#         .filter(
#             (F.col("quality_status") == "VALID") &
#             (F.col("quantity") < F.col("reorder_level"))
#         )
#         .withColumn(
#             "shortage_quantity",
#             F.col("reorder_level") - F.col("quantity")
#         )
#         .select(
#             "product_id",
#             "product_name",
#             "category",
#             "supplier_id",
#             "quantity",
#             "reorder_level",
#             "shortage_quantity"
#         )
#     )

#     (
#         restock_candidates_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable("workspace.default.sentinel_gold_restock")
#     )

#     print("Gold restock table updated")

#     # 6. Generate Gold negative-margin table
#     negative_margin_df = (
#         validated_bronze_df
#         .filter(F.col("selling_price") < F.col("cost_price"))
#         .withColumn(
#             "loss_per_unit",
#             F.col("cost_price") - F.col("selling_price")
#         )
#         .select(
#             "product_id",
#             "product_name",
#             "category",
#             "cost_price",
#             "selling_price",
#             "loss_per_unit"
#         )
#     )

#     (
#         negative_margin_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable("workspace.default.sentinel_gold_negative_margin")
#     )

#     print("Gold negative-margin table updated")

#     # 7. Write Gold quality metrics
#     production_gold_metrics_df = production_metrics_df.withColumn(
#         "processed_at",
#         F.current_timestamp()
#     )

#     (
#         production_gold_metrics_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable("workspace.default.sentinel_gold_quality_metrics")
#     )

#     print("Gold quality metrics updated")

#     # 8. Calculate and save Gold issue metrics
#     issue_metrics_df = (
#         validated_bronze_df
#         .select(F.explode("quality_issues").alias("issue_type"))
#         .groupBy("issue_type")
#         .agg(F.count("*").alias("issue_count"))
#         .orderBy(F.desc("issue_count"))
#     )

#     (
#         issue_metrics_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable("workspace.default.sentinel_gold_issue_metrics")
#     )

#     print("Gold issue metrics updated")

#     # 9. Append historical quality metrics
#     production_history_df = (
#         production_metrics_df
#         .withColumn("run_id", F.lit(pipeline_run_id))
#         .withColumn("run_timestamp", F.current_timestamp())
#         .select(
#             "total_records",
#             "valid_records",
#             "invalid_records",
#             "critical_records",
#             "warning_records",
#             "quality_score",
#             "run_id",
#             "run_timestamp"
#         )
#     )

#     (
#         production_history_df.write
#         .format("delta")
#         .mode("append")
#         .saveAsTable("workspace.default.sentinel_quality_history")
#     )

#     print("Production quality history recorded")

#     # 10. Log successful execution
#     log_pipeline_execution("SUCCESS")

# except Exception as e:

#     print("Pipeline execution failed:", str(e))

#     try:
#         log_pipeline_execution("FAILED", str(e))
#     except Exception as logging_error:
#         print("Failed to write pipeline failure log:", str(logging_error))

#     raise
